<a href="https://colab.research.google.com/github/pzartech/TensorFlow-Examples/blob/master/MRO_Inventory_Status_Logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# MRO Inventory Status Logic
# This script provides functions that model the business rules for assigning
# both Reconciliation and Condition statuses to engine modules in an MRO system.
# A developer can adapt this logic to any programming language and database.

# --- Data Structures (for demonstration) ---
# In a real application, these would be objects or database records.

# Example of an expected part from a manifest
# { "part_number": "PN-123", "serial_number": "SN-ABC" }

# Example of a physically scanned part
# { "part_number": "PN-123", "serial_number": "SN-XYZ", "condition_notes": "Damaged Casing" }

# Example of a part's full data record
# {
#   "part_number": "PN-456",
#   "serial_number": "SN-DEF",
#   "is_installed": True,
#   "inspection_passed": False,
#   "repairable": True,
#   "repair_cost": 80000,
#   "replacement_cost": 100000,
#   "is_quarantined": False,
#   ...
# }


# --- Part 1: Reconciliation Logic (Relevant to MRO Workshop) ---
# This function compares the expected list of parts with the physically found parts.

def reconcile_inventory(expected_manifest, physical_scan):
    """
    Determines the reconciliation status of modules by comparing a manifest
    to a physical scan.

    Args:
        expected_manifest (list of dict): A list of parts expected to be on the engine.
        physical_scan (list of dict): A list of parts physically found on the engine.

    Returns:
        list of dict: A combined list with a reconciliation_status for each item.
    """
    reconciled_list = []

    # Use sets for efficient lookup of serial numbers
    expected_sns = {item['serial_number'] for item in expected_manifest}
    physical_sns = {item['serial_number'] for item in physical_scan}

    # Create dictionaries for quick access to full part data
    expected_map = {item['serial_number']: item for item in expected_manifest}
    physical_map = {item['serial_number']: item for item in physical_scan}

    # --- FORMULA 1: Find Present and Incorrect Parts ---
    # Iterate through all physically scanned parts
    for sn, physical_part in physical_map.items():
        reconciliation_status = ""
        if sn in expected_sns:
            expected_part = expected_map[sn]
            # Check if the part numbers match
            if physical_part['part_number'] == expected_part['part_number']:
                reconciliation_status = "Present / Accounted For"
            else:
                # Same serial number, but different part number
                reconciliation_status = "Incorrect Part Number / WPI"
        else:
            # This part was found but not on the manifest
            reconciliation_status = "Unexpected / Found on Engine (FOE)"

        # Add the physically found part to our final list
        reconciled_list.append({**physical_part, "reconciliation_status": reconciliation_status})


    # --- FORMULA 2: Find Missing Parts ---
    # Iterate through expected parts to see which ones were not found
    missing_sns = expected_sns - physical_sns
    for sn in missing_sns:
        expected_part = expected_map[sn]
        reconciled_list.append({**expected_part, "reconciliation_status": "Missing"})

    return reconciled_list


# --- Part 2: Condition Logic (Relevant to Engine Owner) ---
# This function determines the condition or value status of a single part.

def determine_part_condition(part_data):
    """
    Determines the condition status of a single module based on its properties.

    Args:
        part_data (dict): A dictionary containing the module's data.

    Returns:
        str: The determined condition status string.
    """

    # --- FORMULA 3: Check for Holding Statuses First ---
    if part_data.get("is_quarantined"):
        return "Quarantined"

    if part_data.get("is_awaiting_parts"):
        return "Awaiting Parts (AWP)"

    # --- FORMULA 4: Determine Final Condition based on Inspection and Cost ---
    if not part_data.get("inspection_passed"):
        # If inspection fails, we determine if it's repairable or scrap.

        # The BER (Beyond Economical Repair) formula
        # A common rule is if repair cost is > 75% of replacement cost.
        ber_threshold = 0.75
        if part_data.get("repair_cost", 0) > (part_data.get("replacement_cost", 0) * ber_threshold):
            return "Beyond Economical Repair (BER)"

        if not part_data.get("repairable"):
            return "Scrap (SCRP)"
        else:
            return "Unserviceable - Repairable (REP)"

    # --- FORMULA 5: Determine Serviceable Condition ---
    if part_data.get("inspection_passed"):
        # If inspection passed, it's serviceable. Now we specify what kind.
        if part_data.get("history") == "Overhauled":
            return "Overhauled (OH)"
        elif part_data.get("history") == "Repaired":
            return "Repaired (REP)"
        elif part_data.get("history") == "New":
            return "New (NE)"
        else:
            # Default serviceable status if no other history applies
            return "Serviceable (SV)"

    # Default status if no other conditions are met
    return "As-Removed (AR)"


# --- Example Usage ---

# 1. Reconciliation Example
manifest = [
    { "part_number": "PN-123", "serial_number": "SN-ABC" }, # Will be Present
    { "part_number": "PN-456", "serial_number": "SN-DEF" }, # Will be Missing
    { "part_number": "PN-789", "serial_number": "SN-GHI" }  # Will have wrong PN
]

physical_parts = [
    { "part_number": "PN-123", "serial_number": "SN-ABC" },
    { "part_number": "PN-WRONG", "serial_number": "SN-GHI" },
    { "part_number": "PN-EXTRA", "serial_number": "SN-JKL" } # Unexpected part
]

reconciliation_result = reconcile_inventory(manifest, physical_parts)
print("--- Reconciliation Results ---")
for item in reconciliation_result:
    print(f"SN: {item.get('serial_number', 'N/A')}, PN: {item.get('part_number', 'N/A')}, Status: {item['reconciliation_status']}")

print("\n" + "="*30 + "\n")

# 2. Condition Example
part_to_evaluate = {
  "part_number": "PN-456",
  "serial_number": "SN-DEF",
  "inspection_passed": False,
  "repairable": True,
  "repair_cost": 85000,
  "replacement_cost": 100000,
  "history": "Used"
}

condition_status = determine_part_condition(part_to_evaluate)
print("--- Condition Determination Results ---")
print(f"Part SN-DEF has failed inspection. Its status is: {condition_status}")

part_to_evaluate_2 = {
  "part_number": "PN-123",
  "serial_number": "SN-ABC",
  "inspection_passed": True,
  "history": "Overhauled"
}
condition_status_2 = determine_part_condition(part_to_evaluate_2)
print(f"Part SN-ABC has passed inspection. Its status is: {condition_status_2}")